# 03 — RAG Chain & Model Serving Deployment

**Purpose:** Wire the Vector Search retriever to a Foundation Model API chat model to answer
questions about the insurance data, then deploy that chain as a real Mosaic AI Model Serving
endpoint — this is the "RAG served with Mosaic AI Model Serving" deliverable.

### What This Notebook Does
1. Defines an `mlflow.pyfunc.ChatAgent` that retrieves top-k policy documents and grounds a
   Foundation Model API chat completion in them.
2. Smoke-tests the agent locally, then logs and registers it to Unity Catalog with MLflow.
3. Deploys the registered model via the Agent Framework to a Mosaic AI Model Serving
   endpoint and queries it live to prove it works end-to-end.

### Source & Target
| Direction | Resource |
|-----------|----------|
| Source | `insurance_rag_agent.docs.policy_documents_index` |
| Target | UC model `insurance_rag_agent.agent_tools.insurance_rag_model` + serving endpoint |

> **Prerequisites:** Run `src/02_vector_search_index` first. Deploying consumes Free
> Edition's limited model-serving quota — stop/delete the endpoint after demoing.

---

In [ ]:
%pip install -U mlflow databricks-agents databricks-vectorsearch "databricks-sdk[openai]"
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG               = 'insurance_rag_agent'
VS_ENDPOINT_NAME       = 'insurance_rag_agent_vs_endpoint'
INDEX_NAME             = f'{CATALOG}.docs.policy_documents_index'
# The classic, workspace-wide Foundation Model API name — verified to work via
# w.serving_endpoints.get_open_ai_client() even on workspaces whose Serving page shows a
# "moved to AI Gateway" banner. Avoid the personal-looking alias shown in some "Use this
# model" panels (e.g. '<catalog>.<schema>.<model>_backend') — that alias appears to be scoped
# to the identity that triggered its creation and isn't reliably reachable from a deployed
# model's own service principal, which is what caused this project's credential errors.
CHAT_MODEL_ENDPOINT    = 'databricks-meta-llama-3-3-70b-instruct'
RAG_MODEL_NAME         = f'{CATALOG}.agent_tools.insurance_rag_model'
RAG_ENDPOINT_NAME      = 'insurance_rag_endpoint'
NUM_RESULTS            = 3
SAMPLE_QUESTION        = 'What do the records show about smokers in the southeast region?'

print(f'Index: {INDEX_NAME}')
print(f'Chat model endpoint: {CHAT_MODEL_ENDPOINT}')
print(f'Registered model: {RAG_MODEL_NAME}')

In [0]:
import uuid

import mlflow
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse

mlflow.set_registry_uri('databricks-uc')

---

In [ ]:
class InsuranceRAGAgent(ChatAgent):
    '''Retrieves relevant policy documents, then grounds a Foundation Model API chat
    completion in them. Clients are created inside predict() rather than __init__ so
    the agent instance stays picklable for MLflow logging.'''

    def __init__(self, endpoint_name, index_name, chat_model_endpoint, num_results):
        self.endpoint_name       = endpoint_name
        self.index_name          = index_name
        self.chat_model_endpoint = chat_model_endpoint
        self.num_results         = num_results

    def predict(self, messages, context=None, custom_inputs=None):
        from databricks.sdk import WorkspaceClient
        from databricks.vector_search.client import VectorSearchClient

        question = messages[-1].content

        w            = WorkspaceClient()
        bearer_token = w.config.authenticate()['Authorization'].split(' ', 1)[1]

        # VectorSearchClient() with no args does its OWN credential auto-detection via
        # MLflow's tracking URI, which isn't configured inside a served-model container even
        # though `resources=` is declared below — that mismatch throws "the MLflow tracking
        # URI was set to 'None'". Passing the workspace URL and token explicitly bypasses it.
        vsc = VectorSearchClient(
            workspace_url         = w.config.host,
            personal_access_token = bearer_token,
            disable_notice        = True,
        )
        index = vsc.get_index(endpoint_name=self.endpoint_name, index_name=self.index_name)
        results = index.similarity_search(
            query_text  = question,
            columns     = ['policy_text'],
            num_results = self.num_results,
        )
        context_text = '\n'.join(row[0] for row in results['result']['data_array'])

        # get_open_ai_client() targets the classic /serving-endpoints path with zero manual
        # token plumbing — works here because CHAT_MODEL_ENDPOINT is the workspace-wide
        # Foundation Model name, declared as a `resources=` dependency below.
        openai_client = w.serving_endpoints.get_open_ai_client()
        completion = openai_client.chat.completions.create(
            model    = self.chat_model_endpoint,
            messages = [
                {'role': 'system', 'content': f'Answer using only this context:\n{context_text}'},
                {'role': 'user',   'content': question},
            ],
        )
        answer = completion.choices[0].message.content

        return ChatAgentResponse(messages=[
            ChatAgentMessage(id=str(uuid.uuid4()), role='assistant', content=answer)
        ])


rag_agent = InsuranceRAGAgent(
    endpoint_name       = VS_ENDPOINT_NAME,
    index_name          = INDEX_NAME,
    chat_model_endpoint = CHAT_MODEL_ENDPOINT,
    num_results         = NUM_RESULTS,
)

smoke_test = rag_agent.predict(
    [ChatAgentMessage(id=str(uuid.uuid4()), role='user', content=SAMPLE_QUESTION)]
)
print(smoke_test.messages[-1].content)

---

In [ ]:
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksVectorSearchIndex

# Declaring both dependencies lets Databricks inject scoped, auto-rotated credentials for
# them into the deployed container. This only works because CHAT_MODEL_ENDPOINT is now the
# classic, workspace-wide Foundation Model name — a personal/UC-governed alias here would
# make deployment itself fail with "NOT_FOUND: Dependent serving endpoint ... does not exist."
resources = [
    DatabricksVectorSearchIndex(index_name=INDEX_NAME),
    DatabricksServingEndpoint(endpoint_name=CHAT_MODEL_ENDPOINT),
]

# Pin the served container's requirements to exactly what predict() imports, instead of
# letting log_model() auto-capture this whole notebook's environment. The %pip install cell
# above installs the *full* mlflow package (not mlflow-skinny), which drags in scikit-learn,
# matplotlib, scipy, boto3, google-cloud-storage, docker, gunicorn, flask, and more — none of
# which predict() uses. Auto-capturing all of that makes the serving container build large,
# slow, and more likely to time out or fail; this keeps the build to what's actually needed.
pip_requirements = [
    'mlflow',
    'databricks-vectorsearch',
    'databricks-sdk[openai]',
]

with mlflow.start_run(run_name='insurance_rag_agent'):
    logged_model = mlflow.pyfunc.log_model(
        name                  = 'rag_chain',
        python_model          = rag_agent,
        registered_model_name = RAG_MODEL_NAME,
        resources             = resources,
        pip_requirements      = pip_requirements,
    )

model_version = logged_model.registered_model_version
print(f'Registered {RAG_MODEL_NAME} version {model_version}')

---

In [ ]:
from databricks import agents
from mlflow import MlflowClient

# Deploy whatever is actually newest in Unity Catalog, not whatever `model_version` happens
# to hold in this session — if the logging cell above wasn't the last thing run before this
# one (e.g. after a Python restart), model_version could be stale or undefined, and this
# guarantees the deploy targets the real latest version regardless.
latest_version = max(
    int(mv.version) for mv in MlflowClient().search_model_versions(f"name='{RAG_MODEL_NAME}'")
)
print(f'Latest registered version: {latest_version}')

deployment_info = agents.deploy(
    model_name    = RAG_MODEL_NAME,
    model_version = latest_version,
    endpoint_name = RAG_ENDPOINT_NAME,
    scale_to_zero = True,  # required on Free Edition — no provisioned/always-on throughput
)

print(f'Deployed endpoint: {deployment_info.endpoint_name}')

---

In [ ]:
from mlflow.deployments import get_deploy_client

# insurance_rag_endpoint serves an mlflow.pyfunc.ChatAgent, whose native response schema is
# {"messages": [...]} — not OpenAI's {"choices": [...]}. get_open_ai_client() parses the raw
# response into an OpenAI-shaped object expecting `choices`, which stays None here and breaks
# `.choices[0]`. MLflow's deployments client speaks the agent's actual schema directly.
# Uses RAG_ENDPOINT_NAME (a static config constant) rather than deployment_info.endpoint_name,
# so this cell works even in a fresh session where the deploy cell above hasn't been re-run —
# the endpoint already exists in Databricks once deployed once.
deploy_client = get_deploy_client('databricks')

live_response = deploy_client.predict(
    endpoint = RAG_ENDPOINT_NAME,
    inputs   = {'messages': [{'role': 'user', 'content': SAMPLE_QUESTION}]},
)

print(live_response)
print()
print(live_response['messages'][-1]['content'])